# test-D — C1: standard pipeline: MarkItDown + 1,000-character splitter

answering: **gpt-5.6-luna**, embedding: **text-embedding-3-small**.

Means over 3 repeats of 150 in-scope and 20 out-of-scope questions. Reads `answers-r*.json` and `judge-r*.json` in this folder; nothing here calls an LLM.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent.parent / "code"))  # test/code
import imka_eval as E

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

RUN = "test-D"
rows = E.run_rows(RUN)
in_scope = rows[rows.group != "out_of_scope"]

## Documents

The five documents the run searched (pinned in `documents.json`).

In [2]:
docs = E._read_json(E.run_dir(RUN) / "documents.json")["documents"]
pd.DataFrame([{"document": d["document_name"][:45], "document_id": d["document_id"], "chunks": d.get("chunk_count"), "captioning": (d.get("models") or {}).get("captioning", "-"), "embedding": (d.get("embedding") or {}).get("model", E.DEFAULT_EMBEDDING)} for d in docs])

,document,document_id,chunks,captioning,embedding
0,ABB Manual for Induction Motors and Generator,713054b0-29cd-42b5-a55d-43b46ef5fba9,438,-,text-embedding-3-small
1,EMSA Diesel Generator manual-en.pdf,42b3313c-14e4-43f8-ac2e-9a3380dfaf75,135,-,text-embedding-3-small
2,Kohler Generator Manual.pdf,6988c3b4-5daf-43e2-aeec-b7f3c878cd61,428,-,text-embedding-3-small
3,Nidec Motor Manual.pdf,771c25ba-6f96-4ae7-be58-edb897062094,172,-,text-embedding-3-small
4,Siemens_Manual_Vertical_Induction_Motors_EN.p,8dbee7fc-536c-4a2b-8d87-6d531143e195,165,-,text-embedding-3-small


## Results by question group

In [3]:
E.summary(runs=(RUN,), by="group")

n  hit@1  hit@3  hit@5  reciprocal_rank  correct_pdf@5  correct_pdf@1  candidates_hit  faithfulness  answer_relevancy  context_recall  answer_accuracy  refused
run    group                                                                                                                                                                           
test-D tables          30  0.700  0.833  0.833            0.767          0.967          0.878           0.956         0.911             0.724           0.597            0.536      NaN
       cross_document  30  0.667  0.811  0.889            0.755          0.978          0.811           0.967         0.900             0.785           0.839            0.783      NaN
       topical         30  0.633  0.811  0.867            0.724          0.989          0.889           0.933         0.939             0.775           0.809            0.794      NaN
       paraphrase      35  0.619  0.810  0.848            0.714          0.962          0.905           0.962         0.879             0.693           0.676            0.652      NaN
       figures         25  0.520  0.653  0.667            0.587          0.960          0.787           0.773         0.908             0.331           0.297            0.280      NaN
       out_of_scope    20    NaN    NaN    NaN              NaN            NaN            NaN             NaN           NaN               NaN             NaN              NaN    0.983

In [4]:
E.summary(runs=(RUN,), by=None)

,n,hit@1,hit@3,hit@5,reciprocal_rank,correct_pdf@5,correct_pdf@1,candidates_hit,faithfulness,answer_relevancy,context_recall,answer_accuracy,refused
run,,,,,,,,,,,,,
test-D,170,0.631,0.789,0.827,0.713,0.971,0.858,0.924,0.906,0.674,0.656,0.622,0.983


## Stability across the three repeats

In [5]:
in_scope.groupby("repeat")[["hit@5", "context_recall", "answer_accuracy"]].mean().round(3)

,hit@5,context_recall,answer_accuracy
repeat,,,
1,0.833,0.648,0.618
2,0.807,0.640,0.603
3,0.840,0.680,0.643


## Out-of-scope questions

Share of the 20 out-of-scope questions declined, per repeat.

In [6]:
rows[rows.group == "out_of_scope"].groupby("repeat")["refused"].mean().round(3)

repeat
1    0.95
2    1.00
3    1.00
Name: refused, dtype: float64

## Lowest answer accuracy

The ten in-scope questions with the lowest accuracy, averaged over the repeats, with the first repeat's answer.

In [7]:
answers = {r["question"]: r.get("answer") or "" for r in E.load_answers(RUN, 1)["results"]}
worst = (in_scope.groupby(["question", "group", "document_name"])["answer_accuracy"].mean()
         .sort_values().head(10).reset_index())
worst["document_name"] = worst["document_name"].str[:20]
worst["answer (repeat 1)"] = worst["question"].map(lambda q: answers.get(q, "")[:160])
worst

,question,group,document_name,answer_accuracy,answer (repeat 1)
0,According to the Nidec U.S. MOTORS product ser...,topical,Nidec Motor Manual.p,0.0,The provided context does not include the answ...
1,"As a thumb rule, how often does ABB say an ins...",topical,ABB Manual for Induc,0.0,## ABB frequency\n\nThe provided ABB excerpts ...
2,"After re-greasing a U.S. MOTORS brand motor, h...",cross_document,Nidec Motor Manual.p,0.0,The provided context does **not specify how lo...
3,According to the Siemens typical motor control...,tables,Siemens_Manual_Verti,0.0,## Answer\n\nThe provided context does not con...
4,"For an ABB induction machine, what is the tigh...",tables,ABB Manual for Induc,0.0,## Answer\n\nThe provided ABB context does **n...
5,EMSA generator battery: which hydrometer speci...,paraphrase,EMSA Diesel Generato,0.0,## Answer\n\nThe context does **not state whic...
6,At what winding temperatures should a Siemens ...,paraphrase,Siemens_Manual_Verti,0.0,## Answer\n\nThe supplied context does **not**...
7,At what temperature does the KD27V12 coolant t...,tables,Kohler Generator Man,0.0,The provided context does not specify the KD27...
8,"For an ABB induction machine, what is the tigh...",tables,ABB Manual for Induc,0.0,## Answer\n\nThe provided context does **not s...
9,How many newton-metres should an oiled M20 gra...,paraphrase,ABB Manual for Induc,0.0,## Answer\n\n**The required torque cannot be d...


## Answering cost and time per question

OpenAI models are priced per token; Qwen per second of model working time (llama-server's prompt + generation time) at `E.GPU_PRICE_PER_HOUR`. RunPod queueing, cold starts and job overhead are shown (`runpod_exec_s`, `runpod_delay_s`) but not charged. `wall_s` is the question's end-to-end time as the evaluation saw it.

In [8]:
E.answering_cost((RUN,)).T

run,test-D
answers,510
calls/q,3.0
prompt_tokens/q,2293
completion_tokens/q,407
model_s/q,None
runpod_exec_s/q,None
runpod_delay_s/q,None
wall_s/q (median),7.4
"cost per 1,000 questions ($)",0.95


## Judge spend (Anthropic)

In [9]:
spend = E.judge_spend((RUN,))
spend.groupby("model")[["calls", "input_tokens", "output_tokens", "cost_usd"]].sum()

,calls,input_tokens,output_tokens,cost_usd
model,,,,
claude-haiku-5-5,2760,5597539,712804,0.9162
claude-sonnet-5-5,900,1161553,9000,2.4131
